In [0]:
# Building the Fact Table with Surrogate Keys (SKs) only
from pyspark.sql.functions import col, monotonically_increasing_id, date_format

# Loading existing tables from the workspace catalog
df_silver = spark.table("workspace.silver.cleaned_yellow_taxi")
df_dim_v = spark.table("workspace.gold.dim_vendor")
df_dim_p = spark.table("workspace.gold.dim_payment_type")
df_dim_r = spark.table("workspace.gold.dim_rate_code")

# Building the Fact Table and linking dimensions
df_fact_trips = df_silver \
    .join(df_dim_v, "VendorID", "left") \
    .join(df_dim_p, "payment_type", "left") \
    .join(df_dim_r, "RateCodeID", "left") \
    .withColumn("pickup_date_key", date_format(col("tpep_pickup_datetime"), "yyyyMMdd").cast("int")) \
    .select(
        monotonically_increasing_id().alias("trip_sk"), # The primary surrogate key
        col("vendor_sk"),
        col("payment_type_sk"),
        col("rate_code_sk"),
        col("pickup_date_key").alias("date_key"), # The link to dim_date
        # Measures: Numerical values for aggregation in Power BI
        col("passenger_count"),
        col("trip_distance"),
        col("fare_amount"),
        col("total_amount")
    )

# Overwriting the entire schema to match the new Star Schema structure
df_fact_trips.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.gold.fact_trips")

print(f"Fact Table is ready with Surrogate Keys and {df_fact_trips.count()} rows!")